# Advanced JSON Schema in Python — Problems with Complete Solutions

This notebook is a practice-heavy continuation of an introductory JSON Schema lesson.

## Goals

By the end, you will be able to:

- validate Python objects and JSON documents with **JSON Schema Draft 2020-12**
- build **strict** object schemas with `required` and `additionalProperties`
- validate strings with `minLength`, `pattern`, `enum`, `const`, and `format`
- validate arrays with `items`, `prefixItems`, `contains`, `minContains`, and `uniqueItems`
- model nested data with `$defs` and `$ref`
- express alternatives with `oneOf`, `anyOf`, `allOf`, and `not`
- express conditional business rules with `if` / `then` / `else`
- collect **all validation errors** instead of stopping at the first
- report useful error paths for API clients and tests
- write schema-focused tests
- solve progressively harder schema-design problems

> **Best-practice note:** JSON Schema evolves. This notebook explicitly uses Draft 2020-12 instead of relying on an implicit/default draft.


## 0. Setup

Install `jsonschema` if it is not already installed.

In Jupyter, you can uncomment and run:

```python
%pip install -U jsonschema
```

The examples intentionally keep JSON parsing and schema validation as two separate steps:

1. `json.loads(...)` checks whether text is syntactically valid JSON.
2. `jsonschema` checks whether the resulting Python value conforms to the schema.


In [1]:
from __future__ import annotations

import json
from pprint import pprint

from jsonschema import (
    Draft202012Validator,
    FormatChecker,
)
from jsonschema.exceptions import SchemaError, ValidationError

print("Imports complete.")

Imports complete.


## 1. A Modern Baseline Schema

A good production schema usually declares its draft explicitly and rejects accidental fields unless the API intentionally permits them.


In [2]:
PERSON_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "Person",
    "type": "object",
    "properties": {
        "firstName": {
            "type": "string",
            "minLength": 1
        },
        "middleInitial": {
            "type": ["string", "null"],
            "minLength": 1,
            "maxLength": 1
        },
        "lastName": {
            "type": "string",
            "minLength": 1
        },
        "age": {
            "type": "integer",
            "minimum": 0,
            "maximum": 130
        },
        "eyeColor": {
            "type": "string",
            "enum": [
                "amber", "blue", "brown", "gray",
                "green", "hazel", "red", "violet"
            ]
        }
    },
    "required": ["firstName", "lastName"],
    "additionalProperties": False
}

Draft202012Validator.check_schema(PERSON_SCHEMA)
print("PERSON_SCHEMA is itself a valid Draft 2020-12 schema.")

PERSON_SCHEMA is itself a valid Draft 2020-12 schema.


### Reusable validation helpers

`validate()` raises only one `ValidationError`. For diagnostics, tests, and APIs, `iter_errors()` is often more useful because it can return all failures.


In [3]:
FORMAT_CHECKER = FormatChecker()

def path_to_string(path) -> str:
    """Convert a jsonschema error path into a readable JSON-like path."""
    result = "$"
    for part in path:
        if isinstance(part, int):
            result += f"[{part}]"
        else:
            result += f".{part}"
    return result


def collect_errors(instance, schema, *, check_formats=True):
    """Return all schema errors, sorted into a stable order."""
    validator = Draft202012Validator(
        schema,
        format_checker=FORMAT_CHECKER if check_formats else None,
    )
    errors = list(validator.iter_errors(instance))
    return sorted(
        errors,
        key=lambda e: (list(e.absolute_path), e.validator or "", e.message),
    )


def print_errors(instance, schema, *, check_formats=True):
    errors = collect_errors(instance, schema, check_formats=check_formats)

    if not errors:
        print("VALID")
        return

    print(f"INVALID: {len(errors)} error(s)")
    for i, error in enumerate(errors, start=1):
        print(
            f"{i:>2}. {path_to_string(error.absolute_path)} "
            f"[{error.validator}] {error.message}"
        )


def is_valid(instance, schema, *, check_formats=True) -> bool:
    return not collect_errors(instance, schema, check_formats=check_formats)

In [4]:
valid_person = {
    "firstName": "Ada",
    "middleInitial": None,
    "lastName": "Lovelace",
    "age": 36,
    "eyeColor": "brown"
}

invalid_person = {
    "firstName": "",
    "middleInitial": "AB",
    "lastName": "Lovelace",
    "age": -2,
    "eyeColor": "cyan",
    "nickname": "Enchantress of Numbers"
}

print_errors(valid_person, PERSON_SCHEMA)
print()
print_errors(invalid_person, PERSON_SCHEMA)

VALID

INVALID: 5 error(s)
 1. $ [additionalProperties] Additional properties are not allowed ('nickname' was unexpected)
 2. $.age [minimum] -2 is less than the minimum of 0
 3. $.eyeColor [enum] 'cyan' is not one of ['amber', 'blue', 'brown', 'gray', 'green', 'hazel', 'red', 'violet']
 4. $.firstName [minLength] '' should be non-empty
 5. $.middleInitial [maxLength] 'AB' is too long


## 2. JSON Syntax Errors vs Schema Errors

Invalid JSON text never reaches schema validation.


In [5]:
bad_json_text = '''
{
    "firstName": "Ada",
    "lastName": "Lovelace",
}
'''

try:
    parsed = json.loads(bad_json_text)
except json.JSONDecodeError as exc:
    print("JSON syntax error:")
    print(f"  line={exc.lineno}, column={exc.colno}")
    print(f"  message={exc.msg}")
else:
    print_errors(parsed, PERSON_SCHEMA)

JSON syntax error:
  line=4, column=27
  message=Illegal trailing comma before end of object


---

# Problem 1 — Strict User Registration Schema

Design a schema for this payload:

```json
{
  "username": "alice_42",
  "email": "alice@example.com",
  "age": 24,
  "role": "member"
}
```

Requirements:

- object only
- `username`, `email`, and `role` are required
- username length: 3–20
- username may contain only ASCII letters, digits, and `_`
- email must use the standard `email` format
- age is optional, integer, and between 13 and 120
- role is one of `member`, `moderator`, `admin`
- unknown fields are rejected

Then validate at least four examples.


## Solution 1

In [6]:
USER_REGISTRATION_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "UserRegistration",
    "type": "object",
    "properties": {
        "username": {
            "type": "string",
            "minLength": 3,
            "maxLength": 20,
            "pattern": "^[A-Za-z0-9_]+$"
        },
        "email": {
            "type": "string",
            "format": "email"
        },
        "age": {
            "type": "integer",
            "minimum": 13,
            "maximum": 120
        },
        "role": {
            "type": "string",
            "enum": ["member", "moderator", "admin"]
        }
    },
    "required": ["username", "email", "role"],
    "additionalProperties": False
}

Draft202012Validator.check_schema(USER_REGISTRATION_SCHEMA)

In [7]:
registration_cases = [
    {
        "username": "alice_42",
        "email": "alice@example.com",
        "age": 24,
        "role": "member",
    },
    {
        "username": "x",
        "email": "not-an-email",
        "age": 12,
        "role": "owner",
    },
    {
        "username": "bad name",
        "email": "person@example.org",
        "role": "admin",
    },
    {
        "username": "valid_user",
        "email": "valid@example.org",
        "role": "moderator",
        "debug": True,
    },
]

for i, case in enumerate(registration_cases, start=1):
    print(f"CASE {i}")
    print_errors(case, USER_REGISTRATION_SCHEMA)
    print("-" * 60)

CASE 1
VALID
------------------------------------------------------------
CASE 2
INVALID: 4 error(s)
 1. $.age [minimum] 12 is less than the minimum of 13
 2. $.email [format] 'not-an-email' is not a 'email'
 3. $.role [enum] 'owner' is not one of ['member', 'moderator', 'admin']
 4. $.username [minLength] 'x' is too short
------------------------------------------------------------
CASE 3
INVALID: 1 error(s)
 1. $.username [pattern] 'bad name' does not match '^[A-Za-z0-9_]+$'
------------------------------------------------------------
CASE 4
INVALID: 1 error(s)
 1. $ [additionalProperties] Additional properties are not allowed ('debug' was unexpected)
------------------------------------------------------------


### Best-practice discussion

`format` checking is deliberately supplied through `FormatChecker()`. Do not assume every validator will enforce every format automatically.

Also note that `pattern` searches by regular expression. Anchoring with `^...$` makes the intent explicit for this username rule.


---

# Problem 2 — Nested Address with `$defs` and `$ref`

Create a reusable address definition and reference it from a customer schema.

Requirements for an address:

- `street`, `city`, `postalCode`, and `countryCode` required
- `countryCode` is exactly two uppercase ASCII letters
- reject unknown address fields

Requirements for a customer:

- required `customerId`, `name`, `shippingAddress`
- `customerId` starts with `CUST-` followed by six digits
- optional `billingAddress`
- both addresses use the same reusable schema
- reject unknown top-level fields


## Solution 2

In [8]:
CUSTOMER_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$defs": {
        "address": {
            "type": "object",
            "properties": {
                "street": {"type": "string", "minLength": 1},
                "city": {"type": "string", "minLength": 1},
                "postalCode": {"type": "string", "minLength": 1},
                "countryCode": {
                    "type": "string",
                    "pattern": "^[A-Z]{2}$"
                }
            },
            "required": ["street", "city", "postalCode", "countryCode"],
            "additionalProperties": False
        }
    },
    "type": "object",
    "properties": {
        "customerId": {
            "type": "string",
            "pattern": "^CUST-[0-9]{6}$"
        },
        "name": {"type": "string", "minLength": 1},
        "shippingAddress": {"$ref": "#/$defs/address"},
        "billingAddress": {"$ref": "#/$defs/address"}
    },
    "required": ["customerId", "name", "shippingAddress"],
    "additionalProperties": False
}

Draft202012Validator.check_schema(CUSTOMER_SCHEMA)

In [9]:
customer = {
    "customerId": "CUST-004217",
    "name": "Grace Hopper",
    "shippingAddress": {
        "street": "1 Compiler Way",
        "city": "Arlington",
        "postalCode": "22201",
        "countryCode": "US",
    },
    "billingAddress": {
        "street": "99 COBOL Ave",
        "city": "New York",
        "postalCode": "10001",
        "countryCode": "usa",   # invalid
        "extra": "not allowed"  # invalid
    }
}

print_errors(customer, CUSTOMER_SCHEMA)

INVALID: 2 error(s)
 1. $.billingAddress [additionalProperties] Additional properties are not allowed ('extra' was unexpected)
 2. $.billingAddress.countryCode [pattern] 'usa' does not match '^[A-Z]{2}$'


---

# Problem 3 — Arrays, Uniqueness, and Item Constraints

Model an order:

- `orderId`: `ORD-` + 8 digits
- `items`: array with at least 1 item
- each item has:
  - `sku`: uppercase letters/digits/hyphens, 3–20 chars
  - `quantity`: integer 1–100
  - `unitPrice`: number > 0
- each item is strict
- top-level object is strict
- `tags`: optional array of unique non-empty strings, at most 10 tags

Then validate an order containing several independent mistakes.


## Solution 3

In [10]:
ORDER_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$defs": {
        "lineItem": {
            "type": "object",
            "properties": {
                "sku": {
                    "type": "string",
                    "minLength": 3,
                    "maxLength": 20,
                    "pattern": "^[A-Z0-9-]+$"
                },
                "quantity": {
                    "type": "integer",
                    "minimum": 1,
                    "maximum": 100
                },
                "unitPrice": {
                    "type": "number",
                    "exclusiveMinimum": 0
                }
            },
            "required": ["sku", "quantity", "unitPrice"],
            "additionalProperties": False
        }
    },
    "type": "object",
    "properties": {
        "orderId": {
            "type": "string",
            "pattern": "^ORD-[0-9]{8}$"
        },
        "items": {
            "type": "array",
            "minItems": 1,
            "items": {"$ref": "#/$defs/lineItem"}
        },
        "tags": {
            "type": "array",
            "maxItems": 10,
            "uniqueItems": True,
            "items": {
                "type": "string",
                "minLength": 1
            }
        }
    },
    "required": ["orderId", "items"],
    "additionalProperties": False
}

In [11]:
bad_order = {
    "orderId": "ORD-123",
    "items": [
        {"sku": "abc", "quantity": 0, "unitPrice": 12.50},
        {"sku": "ABC-2", "quantity": 2.5, "unitPrice": 0},
        {"sku": "ABC-3", "quantity": 1, "unitPrice": 4.99, "coupon": "X"},
    ],
    "tags": ["priority", "priority", ""]
}

print_errors(bad_order, ORDER_SCHEMA)

INVALID: 8 error(s)
 1. $.items[0].quantity [minimum] 0 is less than the minimum of 1
 2. $.items[0].sku [pattern] 'abc' does not match '^[A-Z0-9-]+$'
 3. $.items[1].quantity [type] 2.5 is not of type 'integer'
 4. $.items[1].unitPrice [exclusiveMinimum] 0 is less than or equal to the minimum of 0
 5. $.items[2] [additionalProperties] Additional properties are not allowed ('coupon' was unexpected)
 6. $.orderId [pattern] 'ORD-123' does not match '^ORD-[0-9]{8}$'
 7. $.tags [uniqueItems] ['priority', 'priority', ''] has non-unique elements
 8. $.tags[2] [minLength] '' should be non-empty


---

# Problem 4 — `oneOf`: Exactly One Payment Method

A checkout request must contain exactly one valid payment method:

1. card
2. bank transfer
3. store credit

Design the payment object so that each alternative is distinguishable using `const`.

Why `oneOf` instead of `anyOf`?

Because `oneOf` requires **exactly one** branch to match.


## Solution 4

In [12]:
PAYMENT_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "amount": {
            "type": "number",
            "exclusiveMinimum": 0
        },
        "currency": {
            "type": "string",
            "enum": ["USD", "EUR", "GBP"]
        },
        "payment": {
            "oneOf": [
                {
                    "type": "object",
                    "properties": {
                        "type": {"const": "card"},
                        "last4": {
                            "type": "string",
                            "pattern": "^[0-9]{4}$"
                        },
                        "network": {
                            "enum": ["visa", "mastercard", "amex"]
                        }
                    },
                    "required": ["type", "last4", "network"],
                    "additionalProperties": False
                },
                {
                    "type": "object",
                    "properties": {
                        "type": {"const": "bank_transfer"},
                        "iban": {
                            "type": "string",
                            "minLength": 15,
                            "maxLength": 34
                        }
                    },
                    "required": ["type", "iban"],
                    "additionalProperties": False
                },
                {
                    "type": "object",
                    "properties": {
                        "type": {"const": "store_credit"},
                        "accountId": {
                            "type": "string",
                            "pattern": "^SC-[A-Z0-9]{8}$"
                        }
                    },
                    "required": ["type", "accountId"],
                    "additionalProperties": False
                }
            ]
        }
    },
    "required": ["amount", "currency", "payment"],
    "additionalProperties": False
}

In [13]:
payment_cases = [
    {
        "amount": 49.99,
        "currency": "USD",
        "payment": {
            "type": "card",
            "last4": "4242",
            "network": "visa"
        }
    },
    {
        "amount": 49.99,
        "currency": "USD",
        "payment": {
            "type": "card",
            "iban": "DE89370400440532013000"
        }
    },
    {
        "amount": -1,
        "currency": "BTC",
        "payment": {
            "type": "store_credit",
            "accountId": "wrong"
        }
    }
]

for i, case in enumerate(payment_cases, 1):
    print(f"PAYMENT CASE {i}")
    print_errors(case, PAYMENT_SCHEMA)
    print()

PAYMENT CASE 1
VALID

PAYMENT CASE 2
INVALID: 1 error(s)
 1. $.payment [oneOf] {'type': 'card', 'iban': 'DE89370400440532013000'} is not valid under any of the given schemas

PAYMENT CASE 3
INVALID: 3 error(s)
 1. $.amount [exclusiveMinimum] -1 is less than or equal to the minimum of 0
 2. $.currency [enum] 'BTC' is not one of ['USD', 'EUR', 'GBP']
 3. $.payment [oneOf] {'type': 'store_credit', 'accountId': 'wrong'} is not valid under any of the given schemas



### Digging into `oneOf` context

A top-level `oneOf` error can be terse. Its `.context` contains errors from the attempted branches.


In [14]:
bad_payment = payment_cases[1]
errors = collect_errors(bad_payment, PAYMENT_SCHEMA)

for error in errors:
    print("TOP:", path_to_string(error.absolute_path), "-", error.message)
    for suberror in error.context[:8]:
        print(
            "   BRANCH:",
            path_to_string(suberror.absolute_path),
            "-",
            suberror.message
        )

TOP: $.payment - {'type': 'card', 'iban': 'DE89370400440532013000'} is not valid under any of the given schemas
   BRANCH: $.payment - 'last4' is a required property
   BRANCH: $.payment - 'network' is a required property
   BRANCH: $.payment - Additional properties are not allowed ('iban' was unexpected)
   BRANCH: $.payment.type - 'bank_transfer' was expected
   BRANCH: $.payment.type - 'store_credit' was expected
   BRANCH: $.payment - 'accountId' is a required property
   BRANCH: $.payment - Additional properties are not allowed ('iban' was unexpected)


---

# Problem 5 — Conditional Validation with `if` / `then` / `else`

Design an employee schema.

Rules:

- `employmentType` is `employee` or `contractor`
- all records require `name` and `employmentType`
- if `employmentType == "employee"`:
  - `employeeId` is required
  - `salary` is required and must be non-negative
- otherwise (`contractor`):
  - `contractId` is required
  - `hourlyRate` is required and must be positive
- reject unknown properties

A common trap is to put `additionalProperties: false` only inside conditional branches. Keep the set of permitted properties clear at the outer object level.


## Solution 5

In [15]:
EMPLOYMENT_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "name": {"type": "string", "minLength": 1},
        "employmentType": {
            "enum": ["employee", "contractor"]
        },
        "employeeId": {
            "type": "string",
            "pattern": "^EMP-[0-9]{5}$"
        },
        "salary": {
            "type": "number",
            "minimum": 0
        },
        "contractId": {
            "type": "string",
            "pattern": "^CTR-[0-9]{5}$"
        },
        "hourlyRate": {
            "type": "number",
            "exclusiveMinimum": 0
        }
    },
    "required": ["name", "employmentType"],
    "allOf": [
        {
            "if": {
                "properties": {
                    "employmentType": {"const": "employee"}
                },
                "required": ["employmentType"]
            },
            "then": {
                "required": ["employeeId", "salary"],
                "not": {
                    "anyOf": [
                        {"required": ["contractId"]},
                        {"required": ["hourlyRate"]}
                    ]
                }
            },
            "else": {
                "required": ["contractId", "hourlyRate"],
                "not": {
                    "anyOf": [
                        {"required": ["employeeId"]},
                        {"required": ["salary"]}
                    ]
                }
            }
        }
    ],
    "additionalProperties": False
}

In [16]:
employment_cases = [
    {
        "name": "Lin",
        "employmentType": "employee",
        "employeeId": "EMP-01234",
        "salary": 100_000
    },
    {
        "name": "Kai",
        "employmentType": "employee",
        "contractId": "CTR-99999",
        "salary": 50_000
    },
    {
        "name": "Mira",
        "employmentType": "contractor",
        "contractId": "CTR-00012",
        "hourlyRate": 75.0
    },
]

for case in employment_cases:
    pprint(case)
    print_errors(case, EMPLOYMENT_SCHEMA)
    print("-" * 70)

{'employeeId': 'EMP-01234',
 'employmentType': 'employee',
 'name': 'Lin',
 'salary': 100000}
VALID
----------------------------------------------------------------------
{'contractId': 'CTR-99999',
 'employmentType': 'employee',
 'name': 'Kai',
 'salary': 50000}
INVALID: 2 error(s)
 1. $ [not] {'name': 'Kai', 'employmentType': 'employee', 'contractId': 'CTR-99999', 'salary': 50000} should not be valid under {'anyOf': [{'required': ['contractId']}, {'required': ['hourlyRate']}]}
 2. $ [required] 'employeeId' is a required property
----------------------------------------------------------------------
{'contractId': 'CTR-00012',
 'employmentType': 'contractor',
 'hourlyRate': 75.0,
 'name': 'Mira'}
VALID
----------------------------------------------------------------------


---

# Problem 6 — Dependencies Between Fields

Create an account recovery settings schema.

Rules:

- `email` is required and must be an email
- optional `phone`
- optional `smsRecovery`
- if `smsRecovery` appears, `phone` must also appear
- if `phone` appears, it must match a simplified international pattern
- no unknown fields

Use `dependentRequired`.


## Solution 6

In [17]:
RECOVERY_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "email": {
            "type": "string",
            "format": "email"
        },
        "phone": {
            "type": "string",
            "pattern": "^\\+[1-9][0-9]{7,14}$"
        },
        "smsRecovery": {
            "type": "boolean"
        }
    },
    "required": ["email"],
    "dependentRequired": {
        "smsRecovery": ["phone"]
    },
    "additionalProperties": False
}

recovery_cases = [
    {"email": "a@example.com"},
    {"email": "a@example.com", "smsRecovery": True},
    {
        "email": "a@example.com",
        "smsRecovery": True,
        "phone": "+359888123456"
    },
]

for case in recovery_cases:
    print(case)
    print_errors(case, RECOVERY_SCHEMA)
    print()

{'email': 'a@example.com'}
VALID

{'email': 'a@example.com', 'smsRecovery': True}
INVALID: 1 error(s)
 1. $ [dependentRequired] 'phone' is a dependency of 'smsRecovery'

{'email': 'a@example.com', 'smsRecovery': True, 'phone': '+359888123456'}
VALID



---

# Problem 7 — Tuple-like Arrays with `prefixItems`

Suppose an event log stores compact rows as arrays:

```text
[timestamp, level, message]
```

Requirements:

- exactly 3 elements
- element 0: ISO date-time
- element 1: `INFO`, `WARN`, or `ERROR`
- element 2: non-empty string
- no additional items

In Draft 2020-12, tuple validation uses `prefixItems`.


## Solution 7

In [18]:
LOG_ROW_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "array",
    "prefixItems": [
        {"type": "string", "format": "date-time"},
        {"enum": ["INFO", "WARN", "ERROR"]},
        {"type": "string", "minLength": 1},
    ],
    "items": False,
    "minItems": 3,
    "maxItems": 3,
}

log_rows = [
    ["2026-08-07T12:34:56Z", "INFO", "started"],
    ["not-a-time", "DEBUG", ""],
    ["2026-08-07T12:34:56Z", "INFO", "ok", 123],
]

for row in log_rows:
    print(row)
    print_errors(row, LOG_ROW_SCHEMA)
    print()

['2026-08-07T12:34:56Z', 'INFO', 'started']
VALID

['not-a-time', 'DEBUG', '']
INVALID: 2 error(s)
 1. $[1] [enum] 'DEBUG' is not one of ['INFO', 'WARN', 'ERROR']
 2. $[2] [minLength] '' should be non-empty

['2026-08-07T12:34:56Z', 'INFO', 'ok', 123]
INVALID: 2 error(s)
 1. $ [items] Expected at most 3 items but found 1 extra: 123
 2. $ [maxItems] ['2026-08-07T12:34:56Z', 'INFO', 'ok', 123] is too long



---

# Problem 8 — `contains` and `minContains`

A deployment plan is an array of server objects.

Requirement: the plan may contain many server roles, but it must include **at least two** database servers.

Each server:

- has a unique-looking `name` pattern
- has role `web`, `api`, `worker`, or `database`
- is strict

Use `contains` + `minContains`.


## Solution 8

In [19]:
DEPLOYMENT_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "array",
    "minItems": 2,
    "items": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "pattern": "^[a-z][a-z0-9-]{2,30}$"
            },
            "role": {
                "enum": ["web", "api", "worker", "database"]
            }
        },
        "required": ["name", "role"],
        "additionalProperties": False
    },
    "contains": {
        "type": "object",
        "properties": {
            "role": {"const": "database"}
        },
        "required": ["role"]
    },
    "minContains": 2
}

deployment = [
    {"name": "web-01", "role": "web"},
    {"name": "db-01", "role": "database"},
    {"name": "worker-01", "role": "worker"},
]

print_errors(deployment, DEPLOYMENT_SCHEMA)

INVALID: 1 error(s)
 1. $ [minContains] Too few items match the given schema (expected at least 2 but only 1 matched)


### Extension

JSON Schema's `uniqueItems` compares entire array items. It does **not** directly express “unique by one property such as `name`”.

That kind of cross-item uniqueness rule is a good example of a business invariant that may need an additional application-level check.


In [20]:
def duplicate_server_names(servers):
    seen = set()
    duplicates = set()

    for server in servers:
        name = server.get("name")
        if name in seen:
            duplicates.add(name)
        seen.add(name)

    return sorted(duplicates)


deployment_with_duplicate = [
    {"name": "db-01", "role": "database"},
    {"name": "db-01", "role": "database"},
]

print("Schema-valid?", is_valid(deployment_with_duplicate, DEPLOYMENT_SCHEMA))
print("Duplicate names:", duplicate_server_names(deployment_with_duplicate))

Schema-valid? True
Duplicate names: ['db-01']


---

# Problem 9 — Recursive Schemas

Represent a tree node:

```json
{
  "name": "root",
  "children": [
    {"name": "a", "children": []},
    {"name": "b", "children": [{"name": "c"}]}
  ]
}
```

Each node:

- requires a non-empty `name`
- may contain `children`
- `children` is an array of more nodes
- no unknown fields

Use a recursive `$ref`.


## Solution 9

In [21]:
TREE_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$defs": {
        "node": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "minLength": 1
                },
                "children": {
                    "type": "array",
                    "items": {
                        "$ref": "#/$defs/node"
                    }
                }
            },
            "required": ["name"],
            "additionalProperties": False
        }
    },
    "$ref": "#/$defs/node"
}

tree = {
    "name": "root",
    "children": [
        {"name": "a", "children": []},
        {
            "name": "b",
            "children": [
                {"name": "c"},
                {"name": "", "unexpected": True}
            ]
        }
    ]
}

print_errors(tree, TREE_SCHEMA)

INVALID: 2 error(s)
 1. $.children[1].children[1] [additionalProperties] Additional properties are not allowed ('unexpected' was unexpected)
 2. $.children[1].children[1].name [minLength] '' should be non-empty


---

# Problem 10 — API Response Envelope with Success/Error Variants

Design an API response where the response is **either**:

### Success

```json
{
  "ok": true,
  "data": {...}
}
```

### Error

```json
{
  "ok": false,
  "error": {
    "code": "NOT_FOUND",
    "message": "..."
  }
}
```

Rules:

- success must have `ok: true` and `data`
- success must not have `error`
- error must have `ok: false` and an error object
- error must not have `data`
- error `code` is restricted to a known set
- use `oneOf`


## Solution 10

In [22]:
API_RESPONSE_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "oneOf": [
        {
            "type": "object",
            "properties": {
                "ok": {"const": True},
                "data": {}
            },
            "required": ["ok", "data"],
            "additionalProperties": False
        },
        {
            "type": "object",
            "properties": {
                "ok": {"const": False},
                "error": {
                    "type": "object",
                    "properties": {
                        "code": {
                            "enum": [
                                "BAD_REQUEST",
                                "UNAUTHORIZED",
                                "NOT_FOUND",
                                "CONFLICT",
                                "INTERNAL_ERROR"
                            ]
                        },
                        "message": {
                            "type": "string",
                            "minLength": 1
                        }
                    },
                    "required": ["code", "message"],
                    "additionalProperties": False
                }
            },
            "required": ["ok", "error"],
            "additionalProperties": False
        }
    ]
}

api_cases = [
    {"ok": True, "data": {"id": 123}},
    {
        "ok": False,
        "error": {
            "code": "NOT_FOUND",
            "message": "User does not exist"
        }
    },
    {"ok": True, "error": {"code": "NOT_FOUND", "message": "oops"}},
    {"ok": False, "data": {}},
]

for case in api_cases:
    pprint(case)
    print_errors(case, API_RESPONSE_SCHEMA)
    print("=" * 70)

{'data': {'id': 123}, 'ok': True}
VALID
{'error': {'code': 'NOT_FOUND', 'message': 'User does not exist'}, 'ok': False}
VALID
{'error': {'code': 'NOT_FOUND', 'message': 'oops'}, 'ok': True}
INVALID: 1 error(s)
 1. $ [oneOf] {'ok': True, 'error': {'code': 'NOT_FOUND', 'message': 'oops'}} is not valid under any of the given schemas
{'data': {}, 'ok': False}
INVALID: 1 error(s)
 1. $ [oneOf] {'ok': False, 'data': {}} is not valid under any of the given schemas


---

# Problem 11 — Schema Composition with `allOf`

Create a specialized `AdminUser` by combining:

- a reusable base identity object
- an additional admin-specific rule set

This problem demonstrates composition, but also a subtle design issue: combining closed object schemas (`additionalProperties: false`) with `allOf` can be surprising.

A practical Draft 2020-12 solution is to use `unevaluatedProperties: false` at the composed level.


## Solution 11

In [23]:
ADMIN_USER_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$defs": {
        "identity": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "string",
                    "pattern": "^USR-[0-9]{6}$"
                },
                "displayName": {
                    "type": "string",
                    "minLength": 1
                }
            },
            "required": ["id", "displayName"]
        }
    },
    "allOf": [
        {"$ref": "#/$defs/identity"},
        {
            "type": "object",
            "properties": {
                "adminLevel": {
                    "type": "integer",
                    "minimum": 1,
                    "maximum": 5
                },
                "permissions": {
                    "type": "array",
                    "minItems": 1,
                    "uniqueItems": True,
                    "items": {
                        "enum": [
                            "read",
                            "write",
                            "delete",
                            "manage_users"
                        ]
                    }
                }
            },
            "required": ["adminLevel", "permissions"]
        }
    ],
    "unevaluatedProperties": False
}

admin = {
    "id": "USR-000123",
    "displayName": "Root Admin",
    "adminLevel": 5,
    "permissions": ["read", "write", "manage_users"]
}

print_errors(admin, ADMIN_USER_SCHEMA)

VALID


---

# Problem 12 — `propertyNames` and `patternProperties`

Build a metrics object such as:

```json
{
  "cpu.user": 17.2,
  "cpu.system": 4.5,
  "requests.total": 1200
}
```

Requirements:

- property names must contain lowercase letters, digits, `_`, or `.`
- every value must be a non-negative number
- at least one metric required
- no key may start with `_`


## Solution 12

In [24]:
METRICS_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "minProperties": 1,
    "propertyNames": {
        "type": "string",
        "pattern": "^[a-z0-9][a-z0-9_.]*$"
    },
    "patternProperties": {
        "^[a-z0-9][a-z0-9_.]*$": {
            "type": "number",
            "minimum": 0
        }
    },
    "additionalProperties": False
}

metrics_cases = [
    {
        "cpu.user": 17.2,
        "cpu.system": 4.5,
        "requests.total": 1200
    },
    {
        "_hidden": 1,
        "CPU": 2,
        "ok.metric": -3
    }
]

for metrics in metrics_cases:
    print(metrics)
    print_errors(metrics, METRICS_SCHEMA)
    print()

{'cpu.user': 17.2, 'cpu.system': 4.5, 'requests.total': 1200}
VALID

{'_hidden': 1, 'CPU': 2, 'ok.metric': -3}
INVALID: 4 error(s)
 1. $ [additionalProperties] 'CPU', '_hidden' do not match any of the regexes: '^[a-z0-9][a-z0-9_.]*$'
 2. $ [pattern] 'CPU' does not match '^[a-z0-9][a-z0-9_.]*$'
 3. $ [pattern] '_hidden' does not match '^[a-z0-9][a-z0-9_.]*$'
 4. $.ok.metric [minimum] -3 is less than the minimum of 0



---

# Problem 13 — Versioned Configuration with Conditional Rules

A service configuration has versions `1` and `2`.

Common fields:

- `version`
- `serviceName`

Version 1 requires:

- `endpoint`

Version 2 requires:

- `baseUrl`
- `timeoutSeconds` between 1 and 60

A version-1 document must not contain version-2-only fields, and vice versa.

Solve this with `oneOf` and `const`.


## Solution 13

In [25]:
SERVICE_CONFIG_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "oneOf": [
        {
            "type": "object",
            "properties": {
                "version": {"const": 1},
                "serviceName": {
                    "type": "string",
                    "minLength": 1
                },
                "endpoint": {
                    "type": "string",
                    "format": "uri"
                }
            },
            "required": ["version", "serviceName", "endpoint"],
            "additionalProperties": False
        },
        {
            "type": "object",
            "properties": {
                "version": {"const": 2},
                "serviceName": {
                    "type": "string",
                    "minLength": 1
                },
                "baseUrl": {
                    "type": "string",
                    "format": "uri"
                },
                "timeoutSeconds": {
                    "type": "integer",
                    "minimum": 1,
                    "maximum": 60
                }
            },
            "required": [
                "version",
                "serviceName",
                "baseUrl",
                "timeoutSeconds"
            ],
            "additionalProperties": False
        }
    ]
}

configs = [
    {
        "version": 1,
        "serviceName": "legacy",
        "endpoint": "https://api.example.com/v1"
    },
    {
        "version": 2,
        "serviceName": "modern",
        "baseUrl": "https://api.example.com",
        "timeoutSeconds": 30
    },
    {
        "version": 2,
        "serviceName": "broken",
        "endpoint": "https://old.example.com"
    }
]

for config in configs:
    pprint(config)
    print_errors(config, SERVICE_CONFIG_SCHEMA)
    print("-" * 60)

{'endpoint': 'https://api.example.com/v1',
 'serviceName': 'legacy',
 'version': 1}
VALID
------------------------------------------------------------
{'baseUrl': 'https://api.example.com',
 'serviceName': 'modern',
 'timeoutSeconds': 30,
 'version': 2}
VALID
------------------------------------------------------------
{'endpoint': 'https://old.example.com', 'serviceName': 'broken', 'version': 2}
INVALID: 1 error(s)
 1. $ [oneOf] {'version': 2, 'serviceName': 'broken', 'endpoint': 'https://old.example.com'} is not valid under any of the given schemas
------------------------------------------------------------


---

# Problem 14 — Nullable vs Optional

Explain and demonstrate the difference between:

- a property being **optional**
- a property accepting **null**

Then create a profile schema where:

- `nickname` is optional, but if present must be a string
- `middleName` is required, but may be string or `null`


## Solution 14

In [26]:
PROFILE_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "nickname": {
            "type": "string",
            "minLength": 1
        },
        "middleName": {
            "type": ["string", "null"]
        }
    },
    "required": ["middleName"],
    "additionalProperties": False
}

profile_cases = [
    {"middleName": None},              # valid
    {"nickname": "Ace", "middleName": "Lee"},  # valid
    {"nickname": None, "middleName": "Lee"},   # invalid
    {"nickname": "Ace"},              # invalid: middleName absent
]

for case in profile_cases:
    print(case)
    print_errors(case, PROFILE_SCHEMA)
    print()

{'middleName': None}
VALID

{'nickname': 'Ace', 'middleName': 'Lee'}
VALID

{'nickname': None, 'middleName': 'Lee'}
INVALID: 1 error(s)
 1. $.nickname [type] None is not of type 'string'

{'nickname': 'Ace'}
INVALID: 1 error(s)
 1. $ [required] 'middleName' is a required property



**Key rule:** `required` controls **presence**. `type` controls **allowed values**.

Those are independent concepts.


---

# Problem 15 — Build a Better API Validation Function

Write a function that accepts JSON text and a schema and returns a machine-friendly result:

```python
{
    "ok": False,
    "errors": [
        {
            "path": "$.age",
            "validator": "minimum",
            "message": "-4 is less than the minimum of 0"
        }
    ]
}
```

It must distinguish malformed JSON from schema-validation failures.


## Solution 15

In [27]:
def validate_json_document(text: str, schema: dict) -> dict:
    try:
        instance = json.loads(text)
    except json.JSONDecodeError as exc:
        return {
            "ok": False,
            "errors": [
                {
                    "kind": "json_syntax",
                    "path": "$",
                    "line": exc.lineno,
                    "column": exc.colno,
                    "message": exc.msg,
                }
            ],
        }

    schema_errors = collect_errors(instance, schema)

    if schema_errors:
        return {
            "ok": False,
            "errors": [
                {
                    "kind": "schema",
                    "path": path_to_string(error.absolute_path),
                    "validator": error.validator,
                    "message": error.message,
                }
                for error in schema_errors
            ],
        }

    return {
        "ok": True,
        "value": instance,
        "errors": [],
    }

In [28]:
documents = [
    '{"firstName": "Ada", "lastName": "Lovelace", "age": 36}',
    '{"firstName": "", "lastName": "Lovelace", "age": -4}',
    '{"firstName": "Ada",}',
]

for text in documents:
    result = validate_json_document(text, PERSON_SCHEMA)
    pprint(result)
    print("-" * 80)

{'errors': [],
 'ok': True,
 'value': {'age': 36, 'firstName': 'Ada', 'lastName': 'Lovelace'}}
--------------------------------------------------------------------------------
{'errors': [{'kind': 'schema',
             'message': '-4 is less than the minimum of 0',
             'path': '$.age',
             'validator': 'minimum'},
            {'kind': 'schema',
             'message': "'' should be non-empty",
             'path': '$.firstName',
             'validator': 'minLength'}],
 'ok': False}
--------------------------------------------------------------------------------
{'errors': [{'column': 20,
             'kind': 'json_syntax',
             'line': 1,
             'message': 'Illegal trailing comma before end of object',
             'path': '$'}],
 'ok': False}
--------------------------------------------------------------------------------


---

# Problem 16 — Unit Testing Schemas

Write lightweight tests that:

- assert known-good instances are valid
- assert known-bad instances are invalid
- check the schema itself before testing instances
- make failures easy to diagnose


## Solution 16

In [29]:
def assert_valid(instance, schema):
    errors = collect_errors(instance, schema)
    assert not errors, "\n".join(
        f"{path_to_string(e.absolute_path)}: {e.message}"
        for e in errors
    )


def assert_invalid(instance, schema):
    errors = collect_errors(instance, schema)
    assert errors, "Expected instance to be invalid, but it was valid."


Draft202012Validator.check_schema(PERSON_SCHEMA)

assert_valid(
    {"firstName": "Ada", "lastName": "Lovelace"},
    PERSON_SCHEMA,
)

assert_invalid(
    {"firstName": "", "lastName": "Lovelace"},
    PERSON_SCHEMA,
)

assert_invalid(
    {"firstName": "Ada", "lastName": "Lovelace", "extra": 1},
    PERSON_SCHEMA,
)

print("All schema tests passed.")

All schema tests passed.


### Parameterized-style test table

Even without `pytest`, a table of cases makes it easy to expand coverage.


In [30]:
person_test_cases = [
    (
        "minimal valid",
        {"firstName": "Ada", "lastName": "Lovelace"},
        True,
    ),
    (
        "empty first name",
        {"firstName": "", "lastName": "Lovelace"},
        False,
    ),
    (
        "negative age",
        {"firstName": "Ada", "lastName": "Lovelace", "age": -1},
        False,
    ),
    (
        "age is boolean",
        {"firstName": "Ada", "lastName": "Lovelace", "age": True},
        False,
    ),
    (
        "unknown field",
        {"firstName": "Ada", "lastName": "Lovelace", "x": 1},
        False,
    ),
]

for name, instance, expected in person_test_cases:
    actual = is_valid(instance, PERSON_SCHEMA)
    status = "PASS" if actual == expected else "FAIL"
    print(f"{status:4} | {name:20} | expected={expected} actual={actual}")

PASS | minimal valid        | expected=True actual=True
PASS | empty first name     | expected=False actual=False
PASS | negative age         | expected=False actual=False
PASS | age is boolean       | expected=False actual=False
PASS | unknown field        | expected=False actual=False


## Important Python/JSON Edge Case: `bool` and `integer`

Python's `bool` is a subclass of `int`, but `jsonschema`'s JSON type semantics correctly distinguish JSON booleans from JSON integers.

So `True` should not satisfy a JSON Schema `"type": "integer"` rule.


In [31]:
INTEGER_ONLY_SCHEMA = {"type": "integer"}

for value in [1, 0, -3, True, False, 1.5]:
    print(
        f"{value!r:>5} ->",
        "valid" if is_valid(value, INTEGER_ONLY_SCHEMA) else "invalid"
    )

    1 -> valid
    0 -> valid
   -3 -> valid
 True -> invalid
False -> invalid
  1.5 -> invalid


---

# Problem 17 — Advanced Inventory Record

Design a strict inventory item with these requirements:

- `sku`: required, uppercase letters/digits/hyphens
- `name`: required, 2–100 chars
- `category`: one of `hardware`, `software`, `service`
- `price`: non-negative number
- `taxable`: boolean
- `dimensions` is allowed only for `hardware`
- for hardware:
  - `dimensions` required
  - dimensions require positive `width`, `height`, `depth`
  - `weightKg` required and positive
- for software:
  - `downloadUrl` required and URI-formatted
  - `license` required
- for service:
  - `durationMinutes` required and positive integer

Use `oneOf` with category discriminators.


## Solution 17

In [32]:
INVENTORY_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$defs": {
        "baseProperties": {
            "sku": {
                "type": "string",
                "pattern": "^[A-Z0-9-]{3,30}$"
            },
            "name": {
                "type": "string",
                "minLength": 2,
                "maxLength": 100
            },
            "price": {
                "type": "number",
                "minimum": 0
            },
            "taxable": {
                "type": "boolean"
            }
        },
        "dimensions": {
            "type": "object",
            "properties": {
                "width": {"type": "number", "exclusiveMinimum": 0},
                "height": {"type": "number", "exclusiveMinimum": 0},
                "depth": {"type": "number", "exclusiveMinimum": 0},
            },
            "required": ["width", "height", "depth"],
            "additionalProperties": False
        }
    },
    "oneOf": [
        {
            "type": "object",
            "properties": {
                **{
                    "sku": {"type": "string", "pattern": "^[A-Z0-9-]{3,30}$"},
                    "name": {"type": "string", "minLength": 2, "maxLength": 100},
                    "price": {"type": "number", "minimum": 0},
                    "taxable": {"type": "boolean"},
                },
                "category": {"const": "hardware"},
                "dimensions": {"$ref": "#/$defs/dimensions"},
                "weightKg": {"type": "number", "exclusiveMinimum": 0},
            },
            "required": [
                "sku", "name", "category", "price",
                "taxable", "dimensions", "weightKg"
            ],
            "additionalProperties": False
        },
        {
            "type": "object",
            "properties": {
                "sku": {"type": "string", "pattern": "^[A-Z0-9-]{3,30}$"},
                "name": {"type": "string", "minLength": 2, "maxLength": 100},
                "price": {"type": "number", "minimum": 0},
                "taxable": {"type": "boolean"},
                "category": {"const": "software"},
                "downloadUrl": {"type": "string", "format": "uri"},
                "license": {"type": "string", "minLength": 1},
            },
            "required": [
                "sku", "name", "category", "price",
                "taxable", "downloadUrl", "license"
            ],
            "additionalProperties": False
        },
        {
            "type": "object",
            "properties": {
                "sku": {"type": "string", "pattern": "^[A-Z0-9-]{3,30}$"},
                "name": {"type": "string", "minLength": 2, "maxLength": 100},
                "price": {"type": "number", "minimum": 0},
                "taxable": {"type": "boolean"},
                "category": {"const": "service"},
                "durationMinutes": {
                    "type": "integer",
                    "minimum": 1
                },
            },
            "required": [
                "sku", "name", "category", "price",
                "taxable", "durationMinutes"
            ],
            "additionalProperties": False
        }
    ]
}

Draft202012Validator.check_schema(INVENTORY_SCHEMA)

In [33]:
inventory_cases = [
    {
        "sku": "KB-100",
        "name": "Mechanical Keyboard",
        "category": "hardware",
        "price": 89.99,
        "taxable": True,
        "dimensions": {
            "width": 44,
            "height": 3.5,
            "depth": 14
        },
        "weightKg": 1.2
    },
    {
        "sku": "APP-001",
        "name": "Desktop App",
        "category": "software",
        "price": 29.99,
        "taxable": True,
        "downloadUrl": "https://example.com/download/app",
        "license": "per-user"
    },
    {
        "sku": "CONSULT",
        "name": "Consulting",
        "category": "service",
        "price": 150,
        "taxable": False,
        "durationMinutes": 60
    },
    {
        "sku": "BAD-1",
        "name": "Broken",
        "category": "hardware",
        "price": 5,
        "taxable": True,
        "downloadUrl": "https://example.com/file"
    }
]

for item in inventory_cases:
    print(item["sku"])
    print_errors(item, INVENTORY_SCHEMA)
    print("-" * 70)

KB-100
VALID
----------------------------------------------------------------------
APP-001
VALID
----------------------------------------------------------------------
CONSULT
VALID
----------------------------------------------------------------------
BAD-1
INVALID: 1 error(s)
 1. $ [oneOf] {'sku': 'BAD-1', 'name': 'Broken', 'category': 'hardware', 'price': 5, 'taxable': True, 'downloadUrl': 'https://example.com/file'} is not valid under any of the given schemas
----------------------------------------------------------------------


---

# Problem 18 — Advanced Challenge: Event Stream Envelope

Build an event schema with:

- `eventId`: UUID
- `eventType`: one of `user.created`, `user.deleted`, `order.created`
- `occurredAt`: date-time
- `metadata`: strict object with optional `traceId`
- `payload`: shape depends on `eventType`

Payload rules:

### `user.created`
- `userId`: UUID
- `email`: email

### `user.deleted`
- `userId`: UUID
- `reason`: optional non-empty string

### `order.created`
- `orderId`: UUID
- `total`: positive number
- `currency`: exactly 3 uppercase letters

Use `$defs`, `$ref`, `oneOf`, and `const`.


## Solution 18

In [34]:
EVENT_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$defs": {
        "metadata": {
            "type": "object",
            "properties": {
                "traceId": {
                    "type": "string",
                    "minLength": 1
                }
            },
            "additionalProperties": False
        },
        "baseEventProperties": {
            "eventId": {
                "type": "string",
                "format": "uuid"
            },
            "occurredAt": {
                "type": "string",
                "format": "date-time"
            },
            "metadata": {
                "$ref": "#/$defs/metadata"
            }
        }
    },
    "oneOf": [
        {
            "type": "object",
            "properties": {
                "eventId": {"type": "string", "format": "uuid"},
                "eventType": {"const": "user.created"},
                "occurredAt": {"type": "string", "format": "date-time"},
                "metadata": {"$ref": "#/$defs/metadata"},
                "payload": {
                    "type": "object",
                    "properties": {
                        "userId": {"type": "string", "format": "uuid"},
                        "email": {"type": "string", "format": "email"}
                    },
                    "required": ["userId", "email"],
                    "additionalProperties": False
                }
            },
            "required": [
                "eventId", "eventType", "occurredAt",
                "metadata", "payload"
            ],
            "additionalProperties": False
        },
        {
            "type": "object",
            "properties": {
                "eventId": {"type": "string", "format": "uuid"},
                "eventType": {"const": "user.deleted"},
                "occurredAt": {"type": "string", "format": "date-time"},
                "metadata": {"$ref": "#/$defs/metadata"},
                "payload": {
                    "type": "object",
                    "properties": {
                        "userId": {"type": "string", "format": "uuid"},
                        "reason": {"type": "string", "minLength": 1}
                    },
                    "required": ["userId"],
                    "additionalProperties": False
                }
            },
            "required": [
                "eventId", "eventType", "occurredAt",
                "metadata", "payload"
            ],
            "additionalProperties": False
        },
        {
            "type": "object",
            "properties": {
                "eventId": {"type": "string", "format": "uuid"},
                "eventType": {"const": "order.created"},
                "occurredAt": {"type": "string", "format": "date-time"},
                "metadata": {"$ref": "#/$defs/metadata"},
                "payload": {
                    "type": "object",
                    "properties": {
                        "orderId": {"type": "string", "format": "uuid"},
                        "total": {"type": "number", "exclusiveMinimum": 0},
                        "currency": {
                            "type": "string",
                            "pattern": "^[A-Z]{3}$"
                        }
                    },
                    "required": ["orderId", "total", "currency"],
                    "additionalProperties": False
                }
            },
            "required": [
                "eventId", "eventType", "occurredAt",
                "metadata", "payload"
            ],
            "additionalProperties": False
        }
    ]
}

In [35]:
event = {
    "eventId": "8e2af12d-98be-4971-926b-f903859dd770",
    "eventType": "order.created",
    "occurredAt": "2026-08-07T13:00:00Z",
    "metadata": {"traceId": "trace-123"},
    "payload": {
        "orderId": "527f3bdb-a192-478a-b778-2400d7d91650",
        "total": 129.95,
        "currency": "EUR"
    }
}

print_errors(event, EVENT_SCHEMA)

VALID


---

# Problem 19 — Find Every Error and Group by Path

For client-facing APIs it can be useful to group multiple messages by the same path.

Write:

```python
errors_by_path(instance, schema)
```

that returns:

```python
{
    "$.age": [
        "...",
        "..."
    ]
}
```


## Solution 19

In [36]:
from collections import defaultdict

def errors_by_path(instance, schema):
    grouped = defaultdict(list)

    for error in collect_errors(instance, schema):
        grouped[path_to_string(error.absolute_path)].append(error.message)

    return dict(grouped)


problematic_person = {
    "firstName": "",
    "middleInitial": "XYZ",
    "lastName": "",
    "age": -10,
    "eyeColor": "orange",
    "extra": 123
}

pprint(errors_by_path(problematic_person, PERSON_SCHEMA))

{'$': ["Additional properties are not allowed ('extra' was unexpected)"],
 '$.age': ['-10 is less than the minimum of 0'],
 '$.eyeColor': ["'orange' is not one of ['amber', 'blue', 'brown', 'gray', "
                "'green', 'hazel', 'red', 'violet']"],
 '$.firstName': ["'' should be non-empty"],
 '$.lastName': ["'' should be non-empty"],
 '$.middleInitial': ["'XYZ' is too long"]}


---

# Problem 20 — Validate Many Records and Produce a Report

Given a list of records, return:

- total count
- valid count
- invalid count
- row index for each invalid record
- compact error messages

This pattern is useful for import pipelines.


## Solution 20

In [37]:
def validate_batch(records, schema):
    failures = []

    for index, record in enumerate(records):
        errors = collect_errors(record, schema)

        if errors:
            failures.append({
                "index": index,
                "errors": [
                    {
                        "path": path_to_string(error.absolute_path),
                        "validator": error.validator,
                        "message": error.message,
                    }
                    for error in errors
                ],
            })

    return {
        "total": len(records),
        "valid": len(records) - len(failures),
        "invalid": len(failures),
        "failures": failures,
    }


people = [
    {"firstName": "Ada", "lastName": "Lovelace", "age": 36},
    {"firstName": "", "lastName": "Turing"},
    {"firstName": "Alan", "lastName": "Turing", "age": "unknown"},
    {"firstName": "Grace", "lastName": "Hopper", "eyeColor": "brown"},
]

report = validate_batch(people, PERSON_SCHEMA)
pprint(report)

{'failures': [{'errors': [{'message': "'' should be non-empty",
                           'path': '$.firstName',
                           'validator': 'minLength'}],
               'index': 1},
              {'errors': [{'message': "'unknown' is not of type 'integer'",
                           'path': '$.age',
                           'validator': 'type'}],
               'index': 2}],
 'invalid': 2,
 'total': 4,
 'valid': 2}


---

# Mini-Problem Set — Predict Before Running

For each instance below, predict whether it is valid **before** running the cell.


In [38]:
prediction_cases = {
    "A": {"firstName": "A", "lastName": "B", "age": 0},
    "B": {"firstName": "A", "lastName": "B", "middleInitial": ""},
    "C": {"firstName": "A", "lastName": "B", "middleInitial": None},
    "D": {"firstName": "A", "lastName": "B", "age": 130},
    "E": {"firstName": "A", "lastName": "B", "age": 131},
    "F": {"firstName": "A", "lastName": "B", "eyeColor": "green"},
    "G": {"firstName": "A", "lastName": "B", "eyeColor": "Green"},
    "H": {"firstName": "A", "lastName": "B", "x": None},
}

for label, instance in prediction_cases.items():
    print(label, "VALID" if is_valid(instance, PERSON_SCHEMA) else "INVALID")

A VALID
B INVALID
C VALID
D VALID
E INVALID
F VALID
G INVALID
H INVALID


---

# Extra Challenge 1 — Password Policy Schema

Write a schema for a password string with all of these constraints:

- length 12–128
- at least one lowercase letter
- at least one uppercase letter
- at least one digit
- at least one symbol from `!@#$%^&*`

Then test five passwords.

> JSON Schema regular expressions use the regex behavior supported by the validator implementation, so keep portability in mind when schemas cross languages.


## Solution — Extra Challenge 1

In [39]:
PASSWORD_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "string",
    "minLength": 12,
    "maxLength": 128,
    "allOf": [
        {"pattern": "[a-z]"},
        {"pattern": "[A-Z]"},
        {"pattern": "[0-9]"},
        {"pattern": "[!@#$%^&*]"},
    ]
}

passwords = [
    "Short1!",
    "alllowercase123!",
    "ALLUPPERCASE123!",
    "NoDigitsHere!",
    "ValidPass123!",
]

for password in passwords:
    print(
        f"{password!r:24}",
        "VALID" if is_valid(password, PASSWORD_SCHEMA) else "INVALID"
    )

'Short1!'                INVALID
'alllowercase123!'       INVALID
'ALLUPPERCASE123!'       INVALID
'NoDigitsHere!'          INVALID
'ValidPass123!'          VALID


---

# Extra Challenge 2 — Coordinate Pair

Represent `[latitude, longitude]`.

- exactly two items
- latitude: -90 to 90
- longitude: -180 to 180
- both numbers

Use `prefixItems`.


## Solution — Extra Challenge 2

In [40]:
COORDINATE_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "array",
    "prefixItems": [
        {
            "type": "number",
            "minimum": -90,
            "maximum": 90
        },
        {
            "type": "number",
            "minimum": -180,
            "maximum": 180
        }
    ],
    "items": False,
    "minItems": 2,
    "maxItems": 2
}

coordinates = [
    [42.6977, 23.3219],
    [91, 23],
    [42, -181],
    [42],
    [42, 23, 99]
]

for value in coordinates:
    print(value)
    print_errors(value, COORDINATE_SCHEMA)
    print()

[42.6977, 23.3219]
VALID

[91, 23]
INVALID: 1 error(s)
 1. $[0] [maximum] 91 is greater than the maximum of 90

[42, -181]
INVALID: 1 error(s)
 1. $[1] [minimum] -181 is less than the minimum of -180

[42]
INVALID: 1 error(s)
 1. $ [minItems] [42] is too short

[42, 23, 99]
INVALID: 2 error(s)
 1. $ [items] Expected at most 2 items but found 1 extra: 99
 2. $ [maxItems] [42, 23, 99] is too long



---

# Extra Challenge 3 — At Least One Contact Method

A contact record requires `name` and must contain at least one of:

- `email`
- `phone`

It may contain both.

Use `anyOf` with `required`.


## Solution — Extra Challenge 3

In [41]:
CONTACT_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "name": {"type": "string", "minLength": 1},
        "email": {"type": "string", "format": "email"},
        "phone": {
            "type": "string",
            "pattern": "^\\+[1-9][0-9]{7,14}$"
        }
    },
    "required": ["name"],
    "anyOf": [
        {"required": ["email"]},
        {"required": ["phone"]}
    ],
    "additionalProperties": False
}

contacts = [
    {"name": "A", "email": "a@example.com"},
    {"name": "B", "phone": "+359888123456"},
    {
        "name": "C",
        "email": "c@example.com",
        "phone": "+359888123456"
    },
    {"name": "D"},
]

for c in contacts:
    print(c)
    print_errors(c, CONTACT_SCHEMA)
    print()

{'name': 'A', 'email': 'a@example.com'}
VALID

{'name': 'B', 'phone': '+359888123456'}
VALID

{'name': 'C', 'email': 'c@example.com', 'phone': '+359888123456'}
VALID

{'name': 'D'}
INVALID: 1 error(s)
 1. $ [anyOf] {'name': 'D'} is not valid under any of the given schemas



---

# Extra Challenge 4 — Forbid a Dangerous Combination

A feature-flags object has optional booleans:

- `cache`
- `debug`
- `production`

Rule: `debug: true` and `production: true` must **not** occur together.

Use `not`.


## Solution — Extra Challenge 4

In [42]:
FEATURE_FLAGS_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "cache": {"type": "boolean"},
        "debug": {"type": "boolean"},
        "production": {"type": "boolean"},
    },
    "not": {
        "properties": {
            "debug": {"const": True},
            "production": {"const": True},
        },
        "required": ["debug", "production"]
    },
    "additionalProperties": False
}

flag_cases = [
    {"cache": True},
    {"debug": True, "production": False},
    {"debug": False, "production": True},
    {"debug": True, "production": True},
]

for flags in flag_cases:
    print(flags, "->", "VALID" if is_valid(flags, FEATURE_FLAGS_SCHEMA) else "INVALID")

{'cache': True} -> VALID
{'debug': True, 'production': False} -> VALID
{'debug': False, 'production': True} -> VALID
{'debug': True, 'production': True} -> INVALID


---

# Extra Challenge 5 — Custom Business Validation Beyond JSON Schema

JSON Schema validates document structure very well, but not every domain invariant belongs in a schema.

Example: an order line's declared `lineTotal` should equal:

```python
quantity * unitPrice
```

Write a two-stage validator:

1. JSON Schema for shape and types
2. Python for arithmetic consistency


## Solution — Extra Challenge 5

In [43]:
ORDER_LINE_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "quantity": {
            "type": "integer",
            "minimum": 1
        },
        "unitPrice": {
            "type": "number",
            "minimum": 0
        },
        "lineTotal": {
            "type": "number",
            "minimum": 0
        }
    },
    "required": ["quantity", "unitPrice", "lineTotal"],
    "additionalProperties": False
}


def validate_order_line(line):
    schema_errors = collect_errors(line, ORDER_LINE_SCHEMA)

    if schema_errors:
        return [
            f"{path_to_string(e.absolute_path)}: {e.message}"
            for e in schema_errors
        ]

    expected = round(line["quantity"] * line["unitPrice"], 2)
    actual = round(line["lineTotal"], 2)

    if actual != expected:
        return [
            f"$.lineTotal: expected {expected}, got {actual}"
        ]

    return []


order_lines = [
    {"quantity": 2, "unitPrice": 19.99, "lineTotal": 39.98},
    {"quantity": 2, "unitPrice": 19.99, "lineTotal": 40.00},
    {"quantity": 0, "unitPrice": 19.99, "lineTotal": 0},
]

for line in order_lines:
    print(line)
    print(validate_order_line(line))
    print()

{'quantity': 2, 'unitPrice': 19.99, 'lineTotal': 39.98}
[]

{'quantity': 2, 'unitPrice': 19.99, 'lineTotal': 40.0}
['$.lineTotal: expected 39.98, got 40.0']

{'quantity': 0, 'unitPrice': 19.99, 'lineTotal': 0}
['$.quantity: 0 is less than the minimum of 1']



---

# Capstone Problem — E-Commerce Checkout Request

Create a single schema for a checkout request.

## Requirements

### Top level
- required:
  - `requestId` UUID
  - `customer`
  - `items`
  - `shipping`
  - `payment`
- no unknown fields

### Customer
- `id`: `CUST-` + 6 digits
- `email`: email
- both required

### Items
- at least one
- each has:
  - `sku`
  - `quantity`
  - `unitPrice`
- strict items

### Shipping
- one of:
  - standard: `method = "standard"`, `address` required
  - pickup: `method = "pickup"`, `storeId` required
- address should be reusable through `$defs`

### Payment
- one of:
  - card with `token`
  - store credit with `accountId`

### Promo
- optional
- if supplied:
  - `code`: uppercase letters/digits, 4–20 chars

### Additional goal
After schema validation, calculate the order subtotal in Python.


## Capstone Solution

In [44]:
CHECKOUT_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$defs": {
        "address": {
            "type": "object",
            "properties": {
                "line1": {"type": "string", "minLength": 1},
                "line2": {"type": "string"},
                "city": {"type": "string", "minLength": 1},
                "postalCode": {"type": "string", "minLength": 1},
                "countryCode": {
                    "type": "string",
                    "pattern": "^[A-Z]{2}$"
                }
            },
            "required": [
                "line1", "city", "postalCode", "countryCode"
            ],
            "additionalProperties": False
        },
        "item": {
            "type": "object",
            "properties": {
                "sku": {
                    "type": "string",
                    "pattern": "^[A-Z0-9-]{3,30}$"
                },
                "quantity": {
                    "type": "integer",
                    "minimum": 1,
                    "maximum": 100
                },
                "unitPrice": {
                    "type": "number",
                    "minimum": 0
                }
            },
            "required": ["sku", "quantity", "unitPrice"],
            "additionalProperties": False
        }
    },
    "type": "object",
    "properties": {
        "requestId": {
            "type": "string",
            "format": "uuid"
        },
        "customer": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "string",
                    "pattern": "^CUST-[0-9]{6}$"
                },
                "email": {
                    "type": "string",
                    "format": "email"
                }
            },
            "required": ["id", "email"],
            "additionalProperties": False
        },
        "items": {
            "type": "array",
            "minItems": 1,
            "items": {"$ref": "#/$defs/item"}
        },
        "shipping": {
            "oneOf": [
                {
                    "type": "object",
                    "properties": {
                        "method": {"const": "standard"},
                        "address": {"$ref": "#/$defs/address"}
                    },
                    "required": ["method", "address"],
                    "additionalProperties": False
                },
                {
                    "type": "object",
                    "properties": {
                        "method": {"const": "pickup"},
                        "storeId": {
                            "type": "string",
                            "pattern": "^STORE-[0-9]{4}$"
                        }
                    },
                    "required": ["method", "storeId"],
                    "additionalProperties": False
                }
            ]
        },
        "payment": {
            "oneOf": [
                {
                    "type": "object",
                    "properties": {
                        "type": {"const": "card"},
                        "token": {
                            "type": "string",
                            "minLength": 10
                        }
                    },
                    "required": ["type", "token"],
                    "additionalProperties": False
                },
                {
                    "type": "object",
                    "properties": {
                        "type": {"const": "store_credit"},
                        "accountId": {
                            "type": "string",
                            "pattern": "^SC-[A-Z0-9]{8}$"
                        }
                    },
                    "required": ["type", "accountId"],
                    "additionalProperties": False
                }
            ]
        },
        "promo": {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "minLength": 4,
                    "maxLength": 20,
                    "pattern": "^[A-Z0-9]+$"
                }
            },
            "required": ["code"],
            "additionalProperties": False
        }
    },
    "required": [
        "requestId",
        "customer",
        "items",
        "shipping",
        "payment"
    ],
    "additionalProperties": False
}

Draft202012Validator.check_schema(CHECKOUT_SCHEMA)

In [45]:
checkout = {
    "requestId": "692cf4d0-e7b6-4a1d-a951-04eac74f6566",
    "customer": {
        "id": "CUST-000321",
        "email": "buyer@example.com"
    },
    "items": [
        {
            "sku": "KB-100",
            "quantity": 2,
            "unitPrice": 89.99
        },
        {
            "sku": "MOUSE-2",
            "quantity": 1,
            "unitPrice": 39.50
        }
    ],
    "shipping": {
        "method": "standard",
        "address": {
            "line1": "10 Example Street",
            "city": "Sofia",
            "postalCode": "1000",
            "countryCode": "BG"
        }
    },
    "payment": {
        "type": "card",
        "token": "tok_example_123456"
    },
    "promo": {
        "code": "SAVE20"
    }
}

print_errors(checkout, CHECKOUT_SCHEMA)

VALID


In [46]:
def checkout_subtotal(checkout_request):
    errors = collect_errors(checkout_request, CHECKOUT_SCHEMA)

    if errors:
        raise ValueError(
            "Checkout must pass schema validation before calculation."
        )

    return round(
        sum(
            item["quantity"] * item["unitPrice"]
            for item in checkout_request["items"]
        ),
        2
    )


print("Subtotal:", checkout_subtotal(checkout))

Subtotal: 219.48


## Capstone Failure Injection

Now deliberately introduce several errors and verify that all of them can be surfaced together.


In [47]:
broken_checkout = {
    **checkout,
    "customer": {
        "id": "BAD-ID",
        "email": "not-an-email"
    },
    "items": [
        {
            "sku": "bad sku",
            "quantity": 0,
            "unitPrice": -1
        }
    ],
    "shipping": {
        "method": "pickup"
    },
    "payment": {
        "type": "card",
        "token": "short"
    },
    "promo": {
        "code": "bad promo!"
    },
    "unexpected": True
}

print_errors(broken_checkout, CHECKOUT_SCHEMA)

INVALID: 9 error(s)
 1. $ [additionalProperties] Additional properties are not allowed ('unexpected' was unexpected)
 2. $.customer.email [format] 'not-an-email' is not a 'email'
 3. $.customer.id [pattern] 'BAD-ID' does not match '^CUST-[0-9]{6}$'
 4. $.items[0].quantity [minimum] 0 is less than the minimum of 1
 5. $.items[0].sku [pattern] 'bad sku' does not match '^[A-Z0-9-]{3,30}$'
 6. $.items[0].unitPrice [minimum] -1 is less than the minimum of 0
 7. $.payment [oneOf] {'type': 'card', 'token': 'short'} is not valid under any of the given schemas
 8. $.promo.code [pattern] 'bad promo!' does not match '^[A-Z0-9]+$'
 9. $.shipping [oneOf] {'method': 'pickup'} is not valid under any of the given schemas


---

# Best-Practice Checklist

When designing production JSON Schemas:

1. **Declare the schema draft** with `$schema`.
2. **Validate the schema itself** using `Draft202012Validator.check_schema(...)`.
3. Use `required` deliberately; remember that optional and nullable are different.
4. Consider `additionalProperties: false` for API contracts where extra keys should be rejected.
5. Reuse repeated structures with `$defs` and `$ref`.
6. Prefer discriminator-style `const` fields inside `oneOf` branches.
7. Use `FormatChecker` when relying on `format`.
8. Use `iter_errors()` for diagnostics instead of stopping at the first error.
9. Report `absolute_path` to make errors useful to API clients.
10. Test both known-good and known-bad examples.
11. Keep cross-field arithmetic, database lookups, and other domain invariants in application code when JSON Schema is not the right tool.
12. Pin the JSON Schema draft in long-lived contracts.
13. Avoid mixing old-draft syntax with modern draft syntax.
14. Be cautious with regex portability if schemas are consumed in multiple languages.
15. Treat schemas as versioned application artifacts: review, test, and change them intentionally.


# Further Practice — Unsolved Exercises

These are intentionally left without code immediately below them so you can practice independently. Solutions can be built using patterns already demonstrated above.

### A. Product Review
Require `rating` from 1–5, a non-empty `title`, optional `comment`, and either `userId` or `anonymous: true`.

### B. Network Rule
Model either a TCP rule with `port`, or an ICMP rule with `icmpType`. Reject mixed fields.

### C. CSV Import Row
Validate a dictionary representing a normalized CSV row. Then add Python-level validation for uniqueness against a set of previously seen IDs.

### D. Feature Rollout
Require `percentage` from 0–100. If percentage is below 100, require a non-empty `allowList`.

### E. Multi-Region Service
Require at least two region objects. At least one must have `primary: true`. Then enforce “exactly one primary” in Python as an additional business rule.

### F. Search Query DSL
Create a schema for nested query objects supporting `and`, `or`, and leaf comparison nodes. This is another recursive-schema exercise.

### G. Pagination Request
Support either cursor-based pagination or page-number pagination, but not both.

### H. Data Export Job
If `format` is `csv`, allow `delimiter`; if `format` is `json`, forbid it.

### I. Notification
Allow `email`, `sms`, or `push` variants with variant-specific payloads.

### J. API Version Migration
Write a `oneOf` schema that accepts both version 1 and version 2 documents, then write Python code that migrates v1 to v2 after validation.


# End

You now have examples covering basic through advanced JSON Schema patterns, plus reusable Python helpers for validation, testing, diagnostics, batch processing, and application-level business rules.
